# Experiment 2: full case corpus

One DAMICORE object represents one eligible category, while each document preserves
the canonical contextual combinations observed in all distinct `source_hash + category`
cases. Category prevalence is retained.


In [ ]:
from pathlib import Path
from itertools import combinations
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository or one of its subdirectories.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hypotheses.violence_against_women.scripts.experiment_common import (
    BIAS_WARNING_THRESHOLD,
    CASE_BALANCED_WORK_ROOT,
    CASE_FULL_WORK_ROOT,
    CASE_SEEDS,
    COMMON_WORK_ROOT,
    NCD_COLOR_VMAX,
    NORMALIZED_WORK_ROOT,
    RESULTS_ROOT,
    case_execution,
    category_order_from_manifest,
    ensure_artifact_directories,
    load_artifact_manifest,
    load_category_map,
    plot_bias_diagnostics,
    plot_cluster_stability,
    plot_ncd_heatmap,
    plot_support,
    render_tree_artifacts,
    run_damicore_experiment,
    same_cluster_pairs,
    write_json,
    write_common_result_artifacts,
)

ensure_artifact_directories()
manifest = load_artifact_manifest()
category_order = category_order_from_manifest(manifest)
category_map = load_category_map(COMMON_WORK_ROOT / "category-map.csv")
category_support = pd.read_csv(COMMON_WORK_ROOT / "category-support.csv")
category_support = category_support.rename(columns={"support": "support"})
assert category_map["category"].tolist() == category_order
assert set(category_support["category"]) == set(category_order)


## Run DAMICORE and record the common result contract


In [ ]:
case_category_map = pd.read_csv(COMMON_WORK_ROOT / "case-category-map.csv")
case_full_support = (
    case_category_map.loc[case_category_map["regime"] == "case-full", ["category", "case_count"]]
    .rename(columns={"case_count": "support"})
    .sort_values("category")
)
bytes_by_category = (
    case_category_map.loc[case_category_map["regime"] == "case-full"]
    .set_index("category")["bytes"]
    .astype(int)
    .to_dict()
)
assert case_full_support["category"].tolist() == category_order

result = run_damicore_experiment(
    experiment_name="case_full",
    corpus_dir=CASE_FULL_WORK_ROOT / "corpus",
    raw_runs_dir=CASE_FULL_WORK_ROOT / "runs",
    execution=case_execution(),
)
assert result["status"] == "completed", result["preview"]
case_full_result = write_common_result_artifacts(
    result=result,
    category_map=category_map,
    category_order=category_order,
    support=case_full_support,
    bytes_by_category=bytes_by_category,
    output_dir=RESULTS_ROOT / "case_full",
    support_column="support",
    support_label="Distinct cases (log scale)",
    title_prefix="Case-full",
)
display(case_full_result["membership_by_category"])
display(case_full_result["distance"].round(3))


## Interpretation boundary

`case-full` combines contextual similarity with observed category prevalence and
corpus volume. It is not directly comparable to an equal-support regime without the
balanced experiment.
